## Self metadata query

In [1]:
# Ingestion (metadata enriched)
# Preparing the Chroma DB collection

from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(
    model="bge-m3",
    keep_alive=1800,  # 30 minutes
)

In [2]:
uk_with_metadata_collection = Chroma(
    collection_name="uk_with_metadata_collection",
    embedding_function=ollama_embeddings)

uk_with_metadata_collection.reset_collection() #A
#A in case it already exists

In [3]:
# Defining content to be ingested and splitting strategy

from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter( #A
    chunk_size=1000, chunk_overlap=100
)


def split_docs_into_chunks(docs):
    text_docs = html2text_transformer.transform_documents(
        docs) #B
    chunks = text_splitter.split_documents(
        text_docs)

    return chunks


uk_destinations = [
    ("Cornwall", "Cornwall"), ("North_Cornwall", "Cornwall"), 
    ("South_Cornwall", "Cornwall"), ("West_Cornwall", "Cornwall"),
    ("Tintagel", "Cornwall"), ("Newquay", "Cornwall")
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_url_with_metadata = [ #C 
    ( f'{wikivoyage_root_url}/{destination}', destination, region)
    for destination, region in uk_destinations]

#A Instantiate a relatively fine-chunk splitting strategy
#B Transform HTML docs into clean text docs
#C Prepare metadata to be imported: Url, UK Destination and UK Region

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
# Enriching a document with metadata: updating metadata

tintagel_url, tintagel_destination, tintagel_region = uk_destination_url_with_metadata[4]
tintagel_html_loader =AsyncHtmlLoader(tintagel_url)
tintagel_docs = tintagel_html_loader.load()

for doc in tintagel_docs:
    doc.metadata['destination'] = tintagel_destination
    doc.metadata['region'] = tintagel_region
    print(doc.metadata)

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.17it/s]

{'source': 'https://en.wikivoyage.org/wiki/Tintagel', 'title': 'Tintagel – Travel guide at Wikivoyage', 'language': 'en', 'destination': 'Tintagel', 'region': 'Cornwall'}


In [5]:
# Enriching a document with metadata: creating metadata

tintagel_docs_with_metadata = [
    Document(page_content=d.page_content,
             metadata = {
                 'source': tintagel_url,
                 'destination': tintagel_destination,
                 'region': tintagel_region
             })
    for d in tintagel_docs
]

In [6]:
# Enriching the UK destination documents with metadata: creating metadata

for (url, destination, region) in uk_destination_url_with_metadata:
    html_loader = AsyncHtmlLoader(url) #A
    docs =  html_loader.load() #B
    
    docs_with_metadata = [
        Document(page_content=d.page_content,
        metadata = {
            'source': url,
            'destination': destination,
            'region': region})
        for d in docs]
             
    chunks = split_docs_into_chunks(docs_with_metadata)

    print(f'Importing: {destination}')
    uk_with_metadata_collection.add_documents(documents=chunks)
#A Loader for one destination
#B Documents (chunks) related to one destination 

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.33it/s]


Importing: Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.41it/s]


Importing: North_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  1.36it/s]


...

Importing: West_Cornwall


Fetching pages: 100%|##########| 1/1 [00:00<00:00,  3.18it/s]


Importing: Tintagel


Fetching pages: 100%|##########| 1/1 [00:01<00:00,  1.01s/it]


Importing: Newquay


In [7]:
# Searching the collection with a metadata filter explicitly

question =  "Events or festivals"
metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':2, 'filter':{'destination': 'Newquay'}})

result_docs = metadata_retriever.invoke(question)
result_docs

[Document(id='6ae85bd9-2e83-4c2c-8fff-f546ee06481a', metadata={'destination': 'Newquay', 'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay. (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members. (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"),
 Document(id='8e15fce8-45f4-4249-a4dc-d70bad036811', metadata={'source': 'https://en.wikivoyage.org/wiki/Newquay', 'destination': 'Newquay', 'region': 'Cornwall'}, page_content='Hidden categories:\n\n  * Has custom banner\n  * Has map markers\n  * Airport listing\n  * Has mapframe\n  * Do listin

In [8]:
# Generating the self metadata query with the SelfQueryRetriever

from langchain_classic.chains.query_constructor.base import AttributeInfo
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever #A
from langchain_ollama import ChatOllama
#A this requires pip install lark

metadata_field_info = [
    AttributeInfo(
        name="destination",
        description="The specific UK destination to be searched",
        type="string",
    ),
    AttributeInfo(
        name="region",
        description="The name of the UK region to be searched",
        type="string",
    )
]

question = "Tell me about events or festivals in the UK town of Newquay"

llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

self_query_retriever = SelfQueryRetriever.from_llm(
    llm, uk_with_metadata_collection, question, 
    metadata_field_info, verbose=True
)

result_docs = self_query_retriever.invoke(question)
result_docs

[Document(id='6ae85bd9-2e83-4c2c-8fff-f546ee06481a', metadata={'destination': 'Newquay', 'region': 'Cornwall', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay. (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members. (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"),
 Document(id='8e15fce8-45f4-4249-a4dc-d70bad036811', metadata={'source': 'https://en.wikivoyage.org/wiki/Newquay', 'region': 'Cornwall', 'destination': 'Newquay'}, page_content='Hidden categories:\n\n  * Has custom banner\n  * Has map markers\n  * Airport listing\n  * Has mapframe\n  * Do listin

In [9]:
# Generating the self metadata query with a LLM function call

# Query schema
import datetime
from typing import Literal, Optional, Tuple, List

from pydantic import BaseModel, Field
from langchain_classic.chains.query_constructor.ir import (
    Comparator,
    Comparison,
    Operation,
    Operator,
    StructuredQuery,
)
from langchain_classic.retrievers.self_query.chroma import ChromaTranslator


class DestinationSearch(BaseModel):
    """Search over a vector database of tourist destinations."""

    content_search: str = Field(
        "",
        description="""Similarity search query applied 
        to tourist destinations.""",
    )
    destination: str = Field(
        ...,
        description="The specific UK destination to be searched.",
    )
    region: str = Field(
        ...,
        description="The name of the UK region to be searched.",
    )

    def pretty_print(self) -> None:
        for field in self.__fields__:
            if getattr(self, field) is not None and getattr(
                self, field) != getattr(
                self.__fields__[field], "default", None
            ):
                print(f"{field}: {getattr(self, field)}")


def build_filter(destination_search: DestinationSearch):
    comparisons = []

    destination = destination_search.destination #A
    region = destination_search.region #A
    
    if destination and destination != '': #B
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="destination",
                value=destination,
            )
        )
    if region and region != '': #C
        comparisons.append(
            Comparison(
                comparator=Comparator.EQ,
                attribute="region",
                value=region,
            )
        )    

    search_filter = Operation(operator=Operator.AND, arguments=comparisons) #D

    chroma_filter = ChromaTranslator().visit_operation(search_filter) #E
        
    return chroma_filter

#A Get destination and region from the structured query
#B If the destination exists, create an 'equality' operation
#C If the region exists, create an 'equality' operation
#D Create a combined search filter
#E Transform the filter into Chroma format

In [10]:
# Conversion of user question to structured query including metadata filter
from langchain_core.prompts import ChatPromptTemplate

system_message = """You are an expert at converting user 
questions into vector database queries. 
You have access to a database of tourist destinations.
Given a question, return a database query optimized 
to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, 
do not try to rephrase them."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_message),
        ("human", "{question}"),
    ]
)

llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

structured_llm = llm.with_structured_output(DestinationSearch, method="function_calling")
query_generator = prompt | structured_llm

question = "Tell me about events or festivals in the UK town of Newquay"

structured_query =query_generator.invoke(question)
structured_query

DestinationSearch(content_search='events and festivals', destination='Newquay', region='Cornwall')

In [11]:
search_filter = build_filter(structured_query)
search_filter

{'$and': [{'destination': {'$eq': 'Newquay'}},
  {'region': {'$eq': 'Cornwall'}}]}

In [12]:
search_query = structured_query.content_search
search_query

'events and festivals'

In [13]:
metadata_retriever = uk_with_metadata_collection.as_retriever(
    search_kwargs={'k':3, 'filter': search_filter})

answer = metadata_retriever.invoke(search_query)
print(answer)

[Document(id='6ae85bd9-2e83-4c2c-8fff-f546ee06481a', metadata={'source': 'https://en.wikivoyage.org/wiki/Newquay', 'destination': 'Newquay', 'region': 'Cornwall'}, page_content="## Do\n\n[edit]\n\n  * Cornish Film Festival. Held annually for two weeks each November around Newquay. (updated Jan 2024)\n  * 50.415741-5.0914781 Newquay Golf Club, Tower Road, TR7 1LT, ☏ +44 1637 872091, info@newquaygolfclub.co.uk. 9AM-4PM. A semi-private golf club established in 1890. Total yardage Championship: 6141, Men: 5708, and Women: 5364. £31 for non-members. (updated Apr 2019)\n\n### Beaches\n\n[edit]\n\nFistral Beach\n\nNewquay is well known as a surfer's paradise. Therefore it offers plenty of\nbeaches:"), Document(id='121f7f72-812d-468a-ac8f-c791f4500009', metadata={'region': 'Cornwall', 'destination': 'Newquay', 'source': 'https://en.wikivoyage.org/wiki/Newquay'}, page_content='## See\n\n[edit]\n\n  * 50.414578-5.0848411 Blue Reef Aquarium, Towan Promenade, TR7 1DU (right next to Towan beach), ☏

***This is only the retrieval step; you still need to wrap it in a RAG chain***

## Generating a structured SQL query

In [14]:
# Connecting to the UkBooking database
from langchain_community.utilities import SQLDatabase
from langchain_community.tools import QuerySQLDatabaseTool
from langchain_classic.chains import create_sql_query_chain
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

```shell
(.venv) PS C:\Gasym\GitHub Self-Study\AI Agents and Agentic Workflows> sqlite3 --version
3.50.4 2025-07-30 19:33:53 4d8adfb30e03f9cf27f800a2c1ba3c48fb4ca1b08b0f5ed59a4d5ecbf45ealt1 (64-bit)
```

Open your operating system shell, navigate to the code folder, and enter the following command to create the UkBooking database:
```shell
(.venv) PS C:\Gasym\GitHub Self-Study\AI Agents and Agentic Workflows\03-qa-chatbots-with-rag>sqlite3 UkBooking.db
```

This opens the SQLite terminal:
```shell
SQLite version 3.50.4 2025-07-30 19:33:53
Enter ".help" for usage hints.
sqlite>
```

In the SQLite terminal, load the SQL scripts to create and populate the UkBooking database:
```shell
sqlite> .read CreateUkBooking.sql
sqlite> .read PopulateUkBooking.sql
```

To confirm the setup, check for records in the `Offer` table:

In [15]:
db = SQLDatabase.from_uri("sqlite:///UkBooking.db")
print(db.get_usable_table_names())

db.run("SELECT * FROM Offer;")

['Accommodation', 'AccommodationType', 'Booking', 'Customer', 'Destination', 'Offer']


"[(1, 1, 'Summer Special', 0.15, '2024-06-01', '2024-08-31'), (2, 2, 'Weekend Getaway', 0.1, '2024-09-01', '2024-12-31'), (3, 3, 'Early Bird Discount', 0.2, '2024-05-01', '2024-06-30'), (4, 4, 'Stay 3 Nights, Get 1 Free', 0.25, '2024-01-01', '2024-03-31'), (5, 5, 'Historic Stay Offer', 0.1, '2024-04-01', '2024-06-30'), (6, 6, 'Autumn Discount', 0.15, '2024-09-01', '2024-11-30'), (7, 7, 'Cottage Retreat Offer', 0.12, '2024-07-01', '2024-09-30'), (8, 8, 'City Break Deal', 0.08, '2024-10-01', '2024-12-31'), (9, 9, 'Luxury Villa Offer', 0.18, '2024-05-01', '2024-08-31'), (10, 10, 'Spa & Wellness Package', 0.2, '2024-04-01', '2024-07-31')]"

In [16]:
# Generate SQL queries from natural language

llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

sql_query_gen_chain = create_sql_query_chain(llm, db)

response = sql_query_gen_chain.invoke(
    {"question": 
     "Give me some offers for Cardiff, including the hotel name"})
response

'SELECT "Offer"."OfferDescription", "Accommodation"."Name" FROM "Offer" JOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId" JOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId" WHERE "Destination"."Name" = \'Cardiff\' LIMIT 5;'

In [17]:
db.run(response) # may return error (risky)

"[('Early Bird Discount', 'Cardiff Camping')]"

In [18]:
# Executing the SQL query [NOTE: THIS MAY THROW AN ERROR]

sql_query_exec_chain = QuerySQLDatabaseTool(db=db)
sql_query_gen_chain = create_sql_query_chain(llm, db)
chain = sql_query_gen_chain | sql_query_exec_chain
chain.invoke({"question": "Give me some offers for Cardiff, including the hotel name"})

"[('Early Bird Discount', 'Cardiff Camping')]"

In [19]:
# Fixing the SQL format

clean_sql_prompt_template = """You are an expert in SQL Lite. 
You are asked to fix badly formed SQL Lite queries, 
which might contain unneded prefixes or suffixes. 
Given the following unclean SQL statement, 
transform it to a clean, 
executable SQL statement for SQL lite.
Always prefix column names with the table name.
Only return an executable SQL statement which terminates 
with a semicolon. Do not return anything else.
Do not include the language name or symbols like ```.

Unclean SQL: {unclean_sql}"""

clean_sql_prompt = ChatPromptTemplate.from_template(clean_sql_prompt_template)

clean_sql_chain = clean_sql_prompt | llm

full_sql_gen_chain = sql_query_gen_chain | \
   clean_sql_chain | StrOutputParser()

question = """Give me some offers for Cardiff, 
including the discount rate and accommodation name"""

response = full_sql_gen_chain.invoke({"question": question})
response

'SELECT "Offer"."OfferDescription", "Offer"."DiscountRate", "Accommodation"."Name" FROM "Offer" JOIN "Accommodation" ON "Offer"."AccommodationId" = "Accommodation"."AccommodationId" JOIN "Destination" ON "Accommodation"."DestinationId" = "Destination"."DestinationId" WHERE "Destination"."Name" = \'Cardiff\' LIMIT 5;'

In [20]:
# Executing the SQL query
sql_query_exec_chain = QuerySQLDatabaseTool(db=db)

sql_query_gen_and_exec_chain = full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

response = sql_query_gen_and_exec_chain.invoke({"question":question})
response

"[('Early Bird Discount', 0.2, 'Cardiff Camping')]"

## Query router

In [21]:
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableLambda

In [22]:
# Setting up the vector store retriever
tourist_info_retriever_chain = RunnableLambda(
    lambda x: x['question']) \
       | uk_with_metadata_collection.as_retriever(
           search_kwargs={'k':2}) 

# Setting up the relational database retriever (Same as sql_query_gen_and_exec_chain above)
uk_accommodation_retriever_chain =  full_sql_gen_chain \
    | sql_query_exec_chain | StrOutputParser()

In [23]:
# Setting up the query router
class RouteQuery(BaseModel):
    """Route a user question to the most relevant datasource."""

    datasource: Literal["tourist_info_store", 
        "uk_booking_db"] = Field(
        ...,
        description="""Given a user question, 
        route it either to a tourist info vector store 
        or a UK accomodation booking relational database.""",
    )

llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=192, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

structured_llm_router = llm.with_structured_output(
    RouteQuery) #A
#A Structured router which uses LLM function calls

In [24]:
# Setting up the question router chain

system = """You are an expert at routing a user question 
to a tourist info vector store 
or to an UK accommodation booking relational database.
The vector store contains tourist information about UK destinations.
Use the vectorstore for general tourist information questions 
on UK destinations. 
For questions about accommodation availability or booking, 
use the UK Booking database."""
route_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

question_router = route_prompt | structured_llm_router

In [25]:
# Testing the router chain
selected_data_source = question_router.invoke(
    {"question": "Have you got any offers in Brighton?"}
)
print(selected_data_source)

selected_data_source = question_router.invoke(
    {"question": "Where are the best beaches in Cornwall?"}
)
print(selected_data_source)

datasource='uk_booking_db'
datasource='tourist_info_store'


In [26]:
# Setting up the retriever chooser
retriever_chains = {
    'tourist_info_store': tourist_info_retriever_chain,
    'uk_booking_db': uk_accommodation_retriever_chain
}

def retriever_chooser(question):
    selected_data_source = question_router.invoke(
        {"question": question})

    return retriever_chains[selected_data_source.datasource]

chosen = retriever_chooser("""Tell me about events or festivals in the UK town of Newquay""") 
print(chosen)

first=RunnableLambda(...) middle=[] last=VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000204D739FB60>, search_kwargs={'k': 2})


In [27]:
# Setting up the full RAG chain

from langchain_core.runnables import RunnablePassthrough

rag_prompt_template = """
Given a question and some context, answer the question.
If you get a structured context, like a tuple, try to 
infer the meaning of the components: 
typically they refer to accommodation offers, 
and the number is a percentage (0.2 means 20%).
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

def execute_rag_chain(question, chosen_retriever):
    full_rag_chain = (
        {
            "context": {"question": RunnablePassthrough()} 
                | chosen_retriever,#A
            "question": RunnablePassthrough(),#B
        }
        | rag_prompt
        | llm
        | StrOutputParser()
    )

    return full_rag_chain.invoke(question)

#A The context is returned by the retriver after feeding to it the rewritten query
#B This is the original user question

In [28]:
# Executing the full RAG chain

# Question on accommodation offers

question = """Give me offers for Cardiff, including the offer name,
accommodation name, discount rate, start date, and end date.""" 

chosen_retriever = retriever_chooser(question)

answer = execute_rag_chain(question, chosen_retriever)
print(answer)

Offer Name: Early Bird Discount, Accommodation Name: Cardiff Camping, Discount Rate: 20%, Start Date: 2024-05-01, End Date: 2024-06-30


```python
question = """Give me some offers for Cardiff, 
including the accommodation name""" 
```
produces *"The offer for Cardiff is Early Bird Discount at Cardiff Camping."*, as it retrieves only `('Early Bird Discount', 'Cardiff Camping')` and ***neither the discount rate nor validity dates are present*** in the RAG context

In [29]:
# Question on tourist information
question_2 = """Tell me about events or festivals 
in the UK town of Newquay"""

chosen_retriever_2 = retriever_chooser(question_2)

answer2 = execute_rag_chain(question_2, chosen_retriever_2)
print(answer2)

The Cornish Film Festival is held annually for two weeks each November around Newquay.


> **Insight:** 
> - The ***local Gemma/BGE-M3*** pipeline produced a narrower answer because its retrieved context explicitly mentioned only the Cornish Film Festival. 
> - ***GPT-5-nano*** *may have retrieved different chunks or supplemented the answer with internal knowledge*, while the *local result remained more strictly grounded in the available context*. 
> - `A more specific prompt can guide the local model to produce the desired detail and format, provided that the retrieved context contains the supporting information.`

## Retrieval post processing

In [41]:
# RAG Fusion

# Multiple query Generation (same as for MultiQueryRetriver)
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama
from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field


multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task is 
to generate five different versions of the given user 
question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the 
user question, your goal is to help
the user overcome some of the limitations of the 
distance-based similarity search. 
Provide these alternative questions separated by newlines.
Original question: {question}
"""

multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template) 


class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  


questions_parser = LineListOutputParser()

llm = ChatOllama(
    model="gemma4:12b-it-q8_0",
    num_ctx=8192, # overrides Ollama’s default context for this model invocation.
    num_predict=512, # limits summary generation.
    temperature=0,
    reasoning=False,
    # keep_alive=1800, # keeps the model loaded for 30 minutes.
)

multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

In [42]:
# Reciprocal Rank Fusion algorithm
# Based on: https://github.com/Raudaschl/rag-fusion/blob/master/main.py

def reciprocal_rank_fusion(results_groups: #A
                           list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple groups of 
        ranked documents and an optional parameter k used in 
        the Reciprocal Rank Fusion (RRF) formula """

    indexed_results = {} #B
    
    for group_id, results_group in enumerate(
        results_groups): #V
        for local_rank, doc in enumerate(results_group):
            indexed_results[(group_id, local_rank)] = doc
    
    fused_scores = {} #D
    
    for key, doc in indexed_results.items(): #E
        group_id, local_rank = key

        if key not in fused_scores:
            fused_scores[key] = 0 #F
        
        doc_current_score = fused_scores[key]        
        fused_scores[key] += 1 / (local_rank + k) #G

    reranked_results = [ #H
        (indexed_results[key], score)
        for key, score in sorted(fused_scores.items(), 
                                 key=lambda x: x[1], reverse=True)
    ]

    return reranked_results
#A Based on: https://github.com/Raudaschl/rag-fusion/blob/master/main.py                
# B Initialize a dictionary to organize results with an index
# C Index the results by (group_id, local_rank)
# D Initialize a dictionary to hold fused scores for each unique document
# E Iterate through the indexed results
# F Initialize an indexed result with a score of 0 if it has not been processed yet
# G calculate the new document score with the RRF formula
# H rerank the results by RRF score 

In [43]:
retriever = uk_with_metadata_collection.as_retriever(search_kwargs={'k':3})
top_three_results = RunnableLambda(lambda x: x[0:3]) #A

rag_fusion_retrieval_chain =multi_query_gen_chain \
    | retriever.map() | reciprocal_rank_fusion \
    | top_three_results #B
        
docs = rag_fusion_retrieval_chain.invoke(
    {"question": question}) #C
len(docs)
#A select the top three results
#B Full RAG fusion retrieval chain
#C testing the retrieval_chain_rag_fusion chain

3

In [44]:
# Incorporating Rag Fusion into the RAG Chain
rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} | rag_fusion_retrieval_chain,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the step-back question
#B This is the original user question

user_question_1 = "Can you give me some tips for a trip to Brighton?"
user_question_2 = "Can you give me some tips for a trip to Tintagel?"

answer_1 = rag_chain.invoke(user_question_1)
answer_2 = rag_chain.invoke(user_question_2)

print(answer_1)
print()
print(answer_2)

I do not know.

Based on the provided documents, here are some tips for a trip to Tintagel:

**Getting There:**
*   **By Car:** Tintagel is located on the B3263. The southern end links to the A39 near Camelford, and the northern end links via Boscastle.
*   **By Bus:** You can take the First Kernow services 94/95 between Wadebridge and Bude. If you are coming from Bodmin Parkway railway station, you can take the BlueFlash 11A bus to Wadebridge to connect.

**Getting Around:**
*   **Walking:** Tintagel is a small village that can be easily explored on foot.
*   **Parking:** If you are driving, car parking takes coins or can be paid via an app. It is recommended to have coins ready before arrival, as shopkeepers are reluctant to give change for parking.

**Places to Visit and Stay:**
*   **Sightseeing:** The village is best known for its medieval castle, King Arthur's Castle.
*   **Dining/Drinking:** You can visit King Arthur's Arms on Fore Street, which is a traditional pub with extensi